# Virelion-DCCP — FINAL One-Click Runtime Bug Hunt

Run the single code cell below in a fresh Colab runtime. It bootstraps its own checkout, installs validation dependencies only into `/content/dccp-bughunt-deps`, and runs the complete runtime/integration harness in a child process environment.

**It does not upgrade or replace Colab's global packages.**


In [ ]:
# One cell. No previous notebook state is required.
from pathlib import Path
import os
import shutil
import subprocess
import sys
import json

ROOT = Path('/content/Virelion-DCCP-BUGHUNT')
DEPS = Path('/content/dccp-bughunt-deps')

for p in (ROOT, DEPS):
    if p.exists():
        shutil.rmtree(p)

subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    '--disable-pip-version-check', '--no-warn-script-location',
    '--target', str(DEPS),
    'jsonschema', 'pytest', 'pytest-cov', 'coverage',
    'hypothesis', 'ruff==0.16.7'
], check=True)

subprocess.run([
    'git', 'clone', '--branch', 'main', '--depth', '1',
    'https://github.com/Virelion-Biotech/Virelion-DCCP.git', str(ROOT)
], check=True)

runner = ROOT / '_onecell_runtime_bughunt.py'
runner.write_text("\nfrom pathlib import Path\nimport json\nimport os\nimport shutil\nimport subprocess\nimport sys\nimport tempfile\n\nROOT = Path(\"/content/Virelion-DCCP-BUGHUNT\")\nDEPS = Path(\"/content/dccp-bughunt-deps\")\n\nfor p in (ROOT, DEPS):\n    if p.exists():\n        shutil.rmtree(p)\n\nsubprocess.run(\n    [\n        sys.executable, \"-m\", \"pip\", \"install\",\n        \"--disable-pip-version-check\",\n        \"--no-warn-script-location\",\n        \"--target\", str(DEPS),\n        \"jsonschema\", \"pytest\", \"pytest-cov\", \"coverage\",\n        \"hypothesis\", \"ruff==0.16.7\",\n    ],\n    check=True,\n)\n\nsubprocess.run(\n    [\n        \"git\", \"clone\", \"--branch\", \"main\", \"--depth\", \"1\",\n        \"https://github.com/Virelion-Biotech/Virelion-DCCP.git\",\n        str(ROOT),\n    ],\n    check=True,\n)\n\ncommit = subprocess.check_output(\n    [\"git\", \"rev-parse\", \"HEAD\"], cwd=ROOT, text=True\n).strip()\n\nenv = os.environ.copy()\nenv[\"PYTHONPATH\"] = os.pathsep.join([str(DEPS), str(ROOT / \"src\")])\nenv[\"PYTHONNOUSERSITE\"] = \"1\"\n\nprint(\"TESTED COMMIT:\", commit)\nprint(\"ROOT:\", ROOT)\nprint(\"DEPS:\", DEPS)\nprint()\n\nresults = []\n\ndef record(name, ok, detail=\"\"):\n    results.append({\"test\": name, \"ok\": bool(ok), \"detail\": detail})\n    print((\"PASS\" if ok else \"FAIL\") + \" :: \" + name)\n    if detail:\n        print(detail)\n\ndef run_python(code):\n    return subprocess.run(\n        [sys.executable, \"-c\", code],\n        cwd=ROOT,\n        env=env,\n        text=True,\n        capture_output=True,\n    )\n\n# 1. Import identity and every module.\np = run_python(\"import dccp; print(dccp.__file__)\")\nrecord(\n    \"import dccp from checkout\",\n    p.returncode == 0 and str(ROOT / \"src\") in p.stdout,\n    (p.stdout + p.stderr).strip(),\n)\n\nfor path in sorted((ROOT / \"src\" / \"dccp\").glob(\"*.py\")):\n    if path.name == \"__init__.py\":\n        continue\n    module = path.stem\n    p = run_python(f\"import dccp.{module}\")\n    record(\n        f\"import dccp.{module}\",\n        p.returncode == 0,\n        (p.stdout + p.stderr).strip(),\n    )\n\n# 2. Every scenario: load + audit.\nscenario_code = \"\"\"\nfrom pathlib import Path\nfrom dccp.scenario import load_scenario\nfrom dccp.audit import audit_scenario\n\nroot = Path(\"scenarios\")\nfiles = sorted(root.rglob(\"*.json\"))\nprint(\"SCENARIO_COUNT\", len(files))\n\nfor path in files:\n    scenario = load_scenario(path)\n    result = audit_scenario(scenario)\n    print(\n        \"SCENARIO\",\n        path,\n        \"PASS\" if result.passed else \"FAIL\",\n        result.schema_errors,\n        result.policy_errors,\n    )\n    if not result.passed:\n        raise AssertionError(str(path))\n\"\"\"\np = run_python(scenario_code)\nrecord(\"all scenarios load + audit\", p.returncode == 0, (p.stdout + p.stderr).strip())\n\n# 3. Registry/library/challenge-set/bundle integration.\nintegration_code = \"\"\"\nfrom pathlib import Path\nimport json\nimport tempfile\n\nfrom dccp.registry import build_registry, write_registry\nfrom dccp.library import (\n    discover_scenarios,\n    load_library,\n    materialize_challenge_set,\n    write_challenge_set,\n)\nfrom dccp.bundle import build_bundle\n\nroot = Path(\"scenarios\")\nfiles = discover_scenarios(root)\nassert files\n\nwith tempfile.TemporaryDirectory() as td:\n    tmp = Path(td)\n\n    registry = build_registry(root, exclude_paths=[tmp / \"registry.json\"])\n    payload = write_registry(root, tmp / \"registry.json\")\n    assert len(registry) == len(files) == payload[\"n_entries\"]\n\n    library = load_library(root)\n    assert len(library) == len(files)\n\n    challenge = materialize_challenge_set(root)\n    assert challenge[\"n_cases\"] == len(library)\n    assert isinstance(challenge[\"set_hash\"], str)\n    assert len(challenge[\"set_hash\"]) == 64\n\n    write_challenge_set(tmp / \"challenge.json\", root)\n    written = json.loads((tmp / \"challenge.json\").read_text())\n    assert written[\"set_hash\"] == challenge[\"set_hash\"]\n\n    bundle = build_bundle(\n        tmp / \"bundle\",\n        run_id=\"colab-bughunt\",\n        input_files=[files[0], files[0]],\n        producer_version=\"0.3.0\",\n        base_dir=Path(\".\").resolve(),\n    )\n    assert len(bundle[\"inputs\"]) == 1\n    assert not bundle[\"inputs\"][0][\"path\"].startswith(\"/\")\n\"\"\"\np = run_python(integration_code)\nrecord(\"registry/library/challenge/bundle integration\", p.returncode == 0, (p.stdout + p.stderr).strip())\n\n# 4. Edge cases.\nedge_code = \"\"\"\nfrom pathlib import Path\nimport tempfile\nfrom dccp.fingerprint import canonical_json\nfrom dccp.bundle import build_bundle\n\nfor value in (float(\"nan\"), float(\"inf\"), float(\"-inf\")):\n    try:\n        canonical_json({\"x\": value})\n    except (ValueError, TypeError):\n        pass\n    else:\n        raise AssertionError(f\"accepted non-finite value: {value!r}\")\n\nwith tempfile.TemporaryDirectory() as td:\n    root = Path(td)\n    base = root / \"base\"\n    base.mkdir()\n    outside = root / \"outside.txt\"\n    outside.write_text(\"x\")\n\n    try:\n        build_bundle(\n            base / \"bundle\",\n            run_id=\"outside\",\n            input_files=[outside],\n            base_dir=base,\n        )\n    except (ValueError, FileNotFoundError, RuntimeError):\n        pass\n    else:\n        raise AssertionError(\"accepted file outside base_dir\")\n\"\"\"\np = run_python(edge_code)\nrecord(\"numeric/path edge cases\", p.returncode == 0, (p.stdout + p.stderr).strip())\n\n# 5. Exact CI Ruff check and pytest.\nfor label, command in [\n    (\"ruff 0.16.7 check src tests\", [sys.executable, \"-m\", \"ruff\", \"check\", \"src\", \"tests\"]),\n    (\"pytest -q\", [sys.executable, \"-m\", \"pytest\", \"-q\"]),\n]:\n    p = subprocess.run(\n        command,\n        cwd=ROOT,\n        env=env,\n        text=True,\n        capture_output=True,\n    )\n    record(label, p.returncode == 0, (p.stdout + p.stderr).strip()[-10000:])\n\n# 6. CLI paths used by CI.\ncli_commands = [\n    [\"-m\", \"dccp.cli\", \"--version\"],\n    [\"-m\", \"dccp.cli\", \"--help\"],\n    [\"-m\", \"dccp.cli\", \"validate\", \"scenarios/examples/SCENARIO-001.ordinary-mi.json\"],\n    [\"-m\", \"dccp.cli\", \"audit-all\", \"scenarios/examples\"],\n    [\"-m\", \"dccp.cli\", \"registry\", \"--root\", \"scenarios/examples\", \"--output\", \"/tmp/dccp-bughunt-registry.json\"],\n    [\"-m\", \"dccp.cli\", \"release-gate\", \"/tmp/dccp-bughunt-registry.json\", \"--base\", \".\"],\n    [\"-m\", \"dccp.cli\", \"materialize\", \"--root\", \"scenarios/examples\", \"--output\", \"/tmp/dccp-bughunt-challenge.json\"],\n    [\"-m\", \"dccp.cli\", \"map-scores\", \"--scores\", \"inflammatory=0.8,contractile_functional=0.4\", \"--draft-id\", \"SCENARIO-902\", \"-o\", \"/tmp/dccp-bughunt-draft.json\"],\n    [\"-m\", \"dccp.cli\", \"validate\", \"/tmp/dccp-bughunt-draft.json\"],\n    [\"-m\", \"dccp.cli\", \"bundle\", \"--output-dir\", \"/tmp/dccp-bughunt-bundle\", \"--run-id\", \"ci\", \"--input-files\", \"scenarios/examples/SCENARIO-001.ordinary-mi.json\"],\n]\n\nfor command in cli_commands:\n    p = subprocess.run(\n        [sys.executable, *command],\n        cwd=ROOT,\n        env=env,\n        text=True,\n        capture_output=True,\n    )\n    detail = (p.stdout + \"\\n\" + p.stderr).strip()\n    record(\n        \"CLI \" + \" \".join(command),\n        p.returncode == 0,\n        \"\" if p.returncode == 0 else detail[-10000:],\n    )\n\n# 7. Repeated deterministic library loading.\ndeterminism_code = \"\"\"\nfrom pathlib import Path\nfrom dccp.library import load_library\nfrom dccp.provenance import canonical_hash\n\nroot = Path(\"scenarios\")\nobserved = []\n\nfor _ in range(25):\n    lib = load_library(root)\n    observed.append(\n        canonical_hash({\n            \"ids\": sorted(x.scenario.scenario_id for x in lib),\n            \"digests\": sorted(x.digest for x in lib),\n        })\n    )\n\nassert len(set(observed)) == 1\n\"\"\"\np = run_python(determinism_code)\nrecord(\"25 repeated library loads deterministic\", p.returncode == 0, (p.stdout + p.stderr).strip())\n\n# 8. Final result.\nreport = {\n    \"commit\": commit,\n    \"total\": len(results),\n    \"passed\": sum(item[\"ok\"] for item in results),\n    \"failed\": sum(not item[\"ok\"] for item in results),\n    \"results\": results,\n}\nreport_path = ROOT / \"colab_runtime_results.json\"\nreport_path.write_text(json.dumps(report, indent=2) + \"\\n\", encoding=\"utf-8\")\n\nprint()\nprint(\"=== FINAL RESULT ===\")\nprint(json.dumps({\n    \"commit\": report[\"commit\"],\n    \"total\": report[\"total\"],\n    \"passed\": report[\"passed\"],\n    \"failed\": report[\"failed\"],\n}, indent=2))\n\nif report[\"failed\"]:\n    print()\n    print(\"=== FAILURES ===\")\n    for item in results:\n        if not item[\"ok\"]:\n            print(\"\\nTEST:\", item[\"test\"])\n            print(item[\"detail\"])\n\n# Do not raise on a test failure: preserve the complete report for diagnosis.\n", encoding='utf-8')

env = os.environ.copy()
env['PYTHONPATH'] = os.pathsep.join([str(DEPS), str(ROOT / 'src')])
env['PYTHONNOUSERSITE'] = '1'

proc = subprocess.run([
    sys.executable, str(runner)
], cwd=ROOT, env=env, text=True, capture_output=True)

print(proc.stdout)
if proc.stderr:
    print('=== PROCESS STDERR ===\n' + proc.stderr)

report_path = ROOT / 'colab_runtime_results.json'
if report_path.exists():
    report = json.loads(report_path.read_text())
    print('\nREPORT:', json.dumps({
        'commit': report['commit'],
        'total': report['total'],
        'passed': report['passed'],
        'failed': report['failed'],
    }, indent=2))
else:
    raise RuntimeError('Runner crashed before creating a result report.')

print('\nRunner exit code:', proc.returncode)
